[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C22_Reasoning_RL_Course/01_long_cot_rl/01_long_cot_rl.ipynb)

# 01 · Long-CoT 与结果奖励（用 numpy 模拟）

目标：在一个**可枚举**的玩具任务上，从零实现**结果奖励 RL**——策略梯度、基线降方差、温度探索、format 奖励——并**亲手观测**「只奖励最终答对，会检查/重试的长策略如何被选出来（长 CoT 涌现的因果链）」。

路线：玩具任务+精确成功率 → 策略梯度(对拍数值梯度) → 基线降方差 → 完整结果奖励 RL 训练 → 温度/熵 → 长 CoT 涌现实验 → ✏️ 练习 → 📖 答案 → 🧪 GSM8K 奖励胶囊。

> 心智模型：**结果奖励 = 对生成方式做自然选择，适应度 = 最终答对率**。我们不训真模型，只在 softmax 表格策略上把这套算法跑对、对拍解析解。

## 1 · 玩具任务与精确成功率（ground truth）

任务 **凑数**：`N_STEPS` 步，每步从动作集选一个数，累加和 `== TARGET` 即成功（可验证奖励）。
策略是逐步的 softmax 分布。动作少 → 可**枚举所有轨迹**算精确成功率，当对拍基准。

In [ ]:
import numpy as np
from itertools import product
rng = np.random.default_rng(0)

ACTIONS = np.array([0, 1, 2, 3])   # 每步可选的数
N_STEPS = 3
TARGET  = 6
A = len(ACTIONS)

def softmax(z, temp=1.0):
    z = (z / temp)
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def reward(traj):
    '''可验证的结果奖励：动作之和命中 TARGET -> 1 else 0。'''
    return float(sum(ACTIONS[a] for a in traj) == TARGET)

def exact_success(logits, temp=1.0):
    '''枚举所有 A^N_STEPS 条轨迹，按策略概率加权求精确期望回报 J(theta)。'''
    probs = softmax(logits, temp)            # (N_STEPS, A)
    J = 0.0
    for traj in product(range(A), repeat=N_STEPS):
        pt = 1.0
        for t, a in enumerate(traj):
            pt *= probs[t][a]
        J += pt * reward(traj)
    return J

logits0 = np.zeros((N_STEPS, A))             # 均匀策略
J0 = exact_success(logits0)
print(f'均匀策略 精确期望回报 J = {J0:.4f}')
assert 0 < J0 < 1
print('✅ 有了可枚举的精确 J(theta)，作为策略梯度的对拍基准')

## 2 · 策略梯度（REINFORCE）对拍数值梯度

策略梯度恒等式：$\nabla_\theta J=\mathbb{E}_\tau[R(\tau)\sum_t\nabla_\theta\log\pi(a_t|s_t)]$。

我们用蒙特卡洛采样估计它，并和**有限差分数值梯度**对拍——这是验证策略梯度实现正确的金标准。

In [ ]:
def dlogp_dlogits(logits, traj, temp=1.0):
    '''对一条轨迹，返回 d(sum_t log pi(a_t|s_t)) / d logits，形状同 logits。
       softmax 的对数梯度：d log p_a / d z_j = [j==a] - p_j（再除以 temp）。'''
    probs = softmax(logits, temp)
    g = np.zeros_like(logits)
    for t, a in enumerate(traj):
        onehot = np.zeros(A); onehot[a] = 1.0
        g[t] += (onehot - probs[t]) / temp
    return g

def pg_estimate(logits, n_samples=20000, temp=1.0, baseline=0.0):
    '''蒙特卡洛策略梯度估计：mean_tau (R - baseline) * grad_logp。'''
    probs = softmax(logits, temp)
    g = np.zeros_like(logits)
    for _ in range(n_samples):
        traj = [rng.choice(A, p=probs[t]) for t in range(N_STEPS)]
        R = reward(traj)
        g += (R - baseline) * dlogp_dlogits(logits, traj, temp)
    return g / n_samples

def numerical_grad_J(logits, eps=1e-5):
    '''对精确 J(theta) 做有限差分（这是真梯度，不带采样噪声）。'''
    g = np.zeros_like(logits)
    it = np.nditer(logits, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        lp = logits.copy(); lp[i] += eps
        lm = logits.copy(); lm[i] -= eps
        g[i] = (exact_success(lp) - exact_success(lm)) / (2 * eps)
    return g

logits = np.array([[0.5, 0.0, -0.3, 0.2],
                   [0.1, 0.4, 0.0, -0.2],
                   [-0.1, 0.2, 0.3, 0.0]])
g_pg  = pg_estimate(logits, n_samples=40000)
g_num = numerical_grad_J(logits)
print('策略梯度估计 (行0):', np.round(g_pg[0], 4))
print('数值真梯度   (行0):', np.round(g_num[0], 4))
assert np.allclose(g_pg, g_num, atol=0.02), 'MC 策略梯度应逼近数值真梯度'
print('✅ 策略梯度 == 数值梯度（对拍通过）：我们的 REINFORCE 实现正确')

## 3 · 基线降方差：同样无偏，方差小一个量级

减去与动作无关的基线 `b` **不改变梯度期望**（无偏），但能大幅降方差。
我们用「一批样本的平均回报」当基线，对比有/无基线时**单样本梯度估计的方差**。

In [ ]:
def pg_per_sample_grads(logits, n_samples=4000, temp=1.0, baseline=0.0):
    '''返回每个样本的标量梯度投影(沿 logits[0,0] 方向)，用于看方差。'''
    probs = softmax(logits, temp)
    vals = []
    for _ in range(n_samples):
        traj = [rng.choice(A, p=probs[t]) for t in range(N_STEPS)]
        R = reward(traj)
        g = (R - baseline) * dlogp_dlogits(logits, traj, temp)
        vals.append(g[0, 0])
    return np.array(vals)

# 估计该 prompt 的平均回报当基线（= 最优基线 E[R] 的 MC 估计）
probs = softmax(logits)
Rs = [reward([rng.choice(A, p=probs[t]) for t in range(N_STEPS)]) for _ in range(20000)]
b = float(np.mean(Rs))
print(f'用作基线的平均回报 b = {b:.4f}')

no_base = pg_per_sample_grads(logits, baseline=0.0)
with_base = pg_per_sample_grads(logits, baseline=b)
print(f'无基线  单样本梯度: 均值={no_base.mean():+.4f}  方差={no_base.var():.4f}')
print(f'有基线  单样本梯度: 均值={with_base.mean():+.4f}  方差={with_base.var():.4f}')
# 期望(均值)应接近一致（无偏），方差应显著下降
assert abs(no_base.mean() - with_base.mean()) < 0.01, '减基线不应改变期望(无偏)'
assert with_base.var() < no_base.var(), '基线应降低方差'
print('✅ 减基线：期望不变(无偏)、方差下降 —— 稀疏奖励 RL 能否训稳的分水岭')

## 4 · 完整结果奖励 RL：把成功率训上去

把前面拼成一个完整训练循环：采一组样本 → 算回报 → 减组均值基线得优势 → 沿策略梯度上升。
用**精确成功率**监控，验证它**单调上升**。这就是 R1-Zero 训练循环的玩具内核。

In [ ]:
def train_outcome_rl(steps=150, group=128, lr=1.2, temp=1.0, seed=0):
    r = np.random.default_rng(seed)
    logits = np.zeros((N_STEPS, A))
    history = [exact_success(logits, temp)]
    for _ in range(steps):
        probs = softmax(logits, temp)
        trajs, Rs = [], []
        for _ in range(group):
            traj = [r.choice(A, p=probs[t]) for t in range(N_STEPS)]
            trajs.append(traj); Rs.append(reward(traj))
        Rs = np.array(Rs)
        b = Rs.mean()                          # 组均值基线
        grad = np.zeros_like(logits)
        for traj, R in zip(trajs, Rs):
            adv = R - b                        # 优势
            grad += adv * dlogp_dlogits(logits, traj, temp)
        grad /= group
        logits = logits + lr * grad            # 梯度上升
        history.append(exact_success(logits, temp))
    return logits, history

final_logits, hist = train_outcome_rl()
print(f'训练前 成功率 = {hist[0]:.3f}')
print(f'训练后 成功率 = {hist[-1]:.3f}')
print('成功率轨迹(每25步):', [f'{h:.2f}' for h in hist[::25]])
assert hist[-1] > hist[0] + 0.3, '结果奖励 RL 应显著提升成功率'
assert hist[-1] > 0.8, '应学到接近最优'
print('✅ 结果奖励 RL：仅靠 0/1 末端奖励，把成功率从随机训到接近最优')

## 5 · 温度、探索与熵塌缩

温度控制探索：低温→贪心(零探索)，高温→均匀(全探索)。RL 会推高熵→若不约束会**熵塌缩**(过早收敛单一模式、停止探索)。
我们计算策略熵，并演示熵塌缩 + 熵奖励如何缓解。

In [ ]:
def policy_entropy(logits, temp=1.0):
    '''每步分布的平均熵(nats)。'''
    probs = softmax(logits, temp)
    ent = -(probs * np.log(probs + 1e-12)).sum(axis=-1)
    return float(ent.mean())

# 温度对熵的影响（注意：必须用非均匀 logits，均匀分布下温度无效果）
logits_nonunif = np.array([[2.0, 0.5, 0.0, -1.0]] * N_STEPS)
for temp in [0.2, 0.5, 1.0, 3.0]:
    print(f'temp={temp:<4}  熵={policy_entropy(logits_nonunif, temp):.3f} nats  '
          f'(最大熵={np.log(A):.3f})')
assert policy_entropy(logits_nonunif, 0.2) < policy_entropy(logits_nonunif, 3.0), '低温更尖锐(熵更低)'

# 熵塌缩演示：极端贪心 + 大 lr 训练，熵骤降
def train_track_entropy(steps=40, group=32, lr=2.0, temp=0.5, ent_coef=0.0, seed=1):
    r = np.random.default_rng(seed)
    logits = np.zeros((N_STEPS, A))
    ents, succ = [], []
    for _ in range(steps):
        probs = softmax(logits, temp)
        trajs = [[r.choice(A, p=probs[t]) for t in range(N_STEPS)] for _ in range(group)]
        Rs = np.array([reward(tr) for tr in trajs]); b = Rs.mean()
        grad = np.zeros_like(logits)
        for tr, R in zip(trajs, Rs):
            grad += (R - b) * dlogp_dlogits(logits, tr, temp)
        grad /= group
        # 熵奖励：鼓励高熵(对 logits 的熵梯度近似用 -(logp+1) 加权)
        if ent_coef > 0:
            for t in range(N_STEPS):
                pe = softmax(logits, temp)[t]
                grad[t] += ent_coef * (-(np.log(pe + 1e-12) + 1)) * pe / temp
        logits = logits + lr * grad
        ents.append(policy_entropy(logits, temp)); succ.append(exact_success(logits, temp))
    return ents, succ

ent_no, _  = train_track_entropy(ent_coef=0.0)
ent_yes, _ = train_track_entropy(ent_coef=0.3)
print(f'\n无熵奖励: 末端熵={ent_no[-1]:.3f}  (塌缩)')
print(f'有熵奖励: 末端熵={ent_yes[-1]:.3f}  (维持探索)')
assert ent_yes[-1] > ent_no[-1], '熵奖励应缓解熵塌缩'
print('✅ 熵奖励维持探索，缓解熵塌缩（推理 RL 防训练早死的关键）')

## 6 · 长 CoT 涌现：只奖励对错，会检查的策略被选出来

核心实验。两阶段任务：先**猜**一个答案（基础正确率 `p`），再决定**是否花预算检查并可能纠正**（检查能把正确率提到 `p'>p`，但你没直接奖励『检查』这个动作）。

只给**最终对错**奖励。若结果奖励能让『选择检查』的概率上升，就证明了**长 CoT（多花步骤）作为提高正确率的副产品被涌现出来**。

In [ ]:
# 两阶段：动作0=直接交卷(便宜,正确率 p_base), 动作1=检查后交卷(贵,正确率 p_check)
# 我们只奖励『最终答对』(+1) 减去一点点检查成本，观察策略是否学会『选检查』。
def make_emergence_env(p_base, p_check, check_cost=0.0):
    def rollout(choose_check, r):
        if choose_check == 1:
            correct = r.random() < p_check
            return float(correct) - check_cost          # 结果奖励(只看对错) - 成本
        else:
            correct = r.random() < p_base
            return float(correct)
    return rollout

def train_choice(p_base, p_check, check_cost=0.0, steps=200, group=64, lr=0.3, seed=0):
    r = np.random.default_rng(seed)
    env = make_emergence_env(p_base, p_check, check_cost)
    logit = np.array([0.0, 0.0])                 # [不检查, 检查]
    p_check_hist = [softmax(logit[None])[0, 1]]
    for _ in range(steps):
        pr = softmax(logit[None])[0]
        acts = r.choice(2, size=group, p=pr)
        Rs = np.array([env(a, r) for a in acts])
        b = Rs.mean()
        grad = np.zeros(2)
        for a, R in zip(acts, Rs):
            onehot = np.zeros(2); onehot[a] = 1.0
            grad += (R - b) * (onehot - pr)
        logit = logit + lr * grad / group
        p_check_hist.append(softmax(logit[None])[0, 1])
    return p_check_hist

# 难题(p_base 低)，检查很有用(p_check 高)
hard = train_choice(p_base=0.3, p_check=0.75)
# 检查没用(p_check≈p_base)：长 CoT 不该涌现
useless = train_choice(p_base=0.6, p_check=0.6)
print(f'难题+有效检查: P(选检查) {hard[0]:.2f} -> {hard[-1]:.2f}  (涌现『多想』)')
print(f'检查无效      : P(选检查) {useless[0]:.2f} -> {useless[-1]:.2f}  (不涌现)')
assert hard[-1] > 0.7, '检查有效时，结果奖励应推高『选检查』(长 CoT 涌现)'
assert useless[-1] < hard[-1], '检查无效时不该涌现长 CoT'
print('✅ 长 CoT 涌现：仅凭对错奖励，『会检查/多想』因提高正确率被选出来；检查无用则不涌现')

---
## ✏️ 练习 1：format 奖励

实现 `total_reward(output, gold)`：正确性奖励(+1，`<answer>` 内答案==gold) + 格式奖励(+0.1，输出恰含一对 `<think></think>` 和一对 `<answer></answer>`)。

格式分远小于正确性分（只为让答案可抽取，不喧宾夺主）。

In [ ]:
import re
def total_reward(output, gold, format_bonus=0.1):
    # TODO: 1) 用正则抽 <answer>...</answer> 里的整数, ==gold 给 +1
    #       2) 恰好一对 <think></think> 且 恰好一对 <answer></answer> 给 +format_bonus
    #       返回二者之和
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
good = '<think>2+2=4 then 4+4=8</think><answer>8</answer>'
wrong_ans = '<think>...</think><answer>9</answer>'
no_fmt = 'the answer is 8'
assert abs(total_reward(good, 8) - 1.1) < 1e-9, '对+格式 = 1.1'
assert abs(total_reward(wrong_ans, 8) - 0.1) < 1e-9, '错但格式对 = 0.1'
assert abs(total_reward(no_fmt, 8) - 0.0) < 1e-9, '无格式无可抽答案 = 0'
# 双 answer 标签不算合法格式
bad_fmt = '<think>x</think><answer>8</answer><answer>8</answer>'
assert total_reward(bad_fmt, 8) == 1.0, '答案对(+1)但格式不合法(无0.1)'
print('✅ 练习 1 通过：正确性主导 + 轻格式奖励')

## ✏️ 练习 2：组优势（GRPO 的前身）

实现 `group_advantage(rewards, normalize=False)`：减组均值（`normalize=False`）或再除组标准差（`normalize=True`）。

验证：减均值后**和为 0**（合法基线）；归一化后均值≈0、标准差≈1。这是模块 02 GRPO 优势的雏形。

In [ ]:
def group_advantage(rewards, normalize=False, eps=1e-8):
    # TODO: adv = r - mean(r); 若 normalize 再 /(std(r)+eps)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
r = np.array([1.0, 0.0, 1.0, 0.0, 1.0])
adv = group_advantage(r)
assert abs(adv.sum()) < 1e-9, '减均值后优势和应为 0(合法基线)'
advn = group_advantage(r, normalize=True)
assert abs(advn.mean()) < 1e-7 and abs(advn.std() - 1.0) < 1e-2, '归一化后均值0方差1'
# 全对组：减均值后全 0（没有相对信号）
assert np.allclose(group_advantage(np.ones(4)), 0.0), '全对组优势全 0'
print('✅ 练习 2 通过：组归一化优势 —— 天然零均值的合法基线')

## ✏️ 练习 3：温度采样

实现 `temperature_sample(logits_1d, temp, r)`：按温度缩放的 softmax 采一个动作。

验证：极低温≈贪心(几乎总选最大 logit)；极高温≈均匀(各动作频率接近)。

In [ ]:
def temperature_sample(logits_1d, temp, r):
    # TODO: p = softmax(logits/temp); 用 r.choice 按 p 采一个动作下标
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
r = np.random.default_rng(0)
lg = np.array([2.0, 0.0, 0.0, 0.0])   # 动作 0 logit 最高
# 低温：几乎总选动作 0
lo = [temperature_sample(lg, 0.1, r) for _ in range(2000)]
assert np.mean(np.array(lo) == 0) > 0.95, '低温应近贪心(几乎总选最大)'
# 高温：趋于均匀
hi = [temperature_sample(lg, 50.0, r) for _ in range(8000)]
freqs = np.bincount(hi, minlength=4) / len(hi)
assert freqs.max() - freqs.min() < 0.1, '高温应趋于均匀'
print('✅ 练习 3 通过：温度是探索-利用旋钮，低温贪心、高温均匀')

## ✏️ 练习 4：策略梯度一步更新

实现 `reinforce_step(logits, trajs, rewards, lr, use_baseline)`：用给定一批轨迹做一步 REINFORCE 更新（可选组均值基线），返回新 logits。

用第 2 节的 `dlogp_dlogits`。验证：用基线时更新更稳（这里只验证形状与方向正确）。

In [ ]:
def reinforce_step(logits, trajs, rewards, lr=0.3, use_baseline=True):
    # TODO: b = mean(rewards) if use_baseline else 0
    #       grad = mean_i (r_i - b) * dlogp_dlogits(logits, traj_i)
    #       return logits + lr * grad
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
r = np.random.default_rng(2)
lg = np.zeros((N_STEPS, A))
pr = softmax(lg)
trajs = [[r.choice(A, p=pr[t]) for t in range(N_STEPS)] for _ in range(64)]
Rs = np.array([reward(tr) for tr in trajs])
lg2 = reinforce_step(lg, trajs, Rs, lr=0.5, use_baseline=True)
assert lg2.shape == lg.shape
# 一步更新应朝提升期望回报方向(精确 J 不降，允许采样噪声小幅波动)
improved = sum(exact_success(reinforce_step(lg, 
    [[r.choice(A, p=pr[t]) for t in range(N_STEPS)] for _ in range(64)],
    np.array([reward(tr) for tr in [[r.choice(A,p=pr[t]) for t in range(N_STEPS)] for _ in range(64)]]),
    lr=0.5)) >= exact_success(lg) - 0.05 for _ in range(5))
assert improved >= 4, '多数情况下一步 REINFORCE 不应让精确成功率明显下降'
print('✅ 练习 4 通过：一步 REINFORCE 更新（含基线）实现正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
import re
def total_reward(output, gold, format_bonus=0.1):
    rew = 0.0
    m = re.findall(r'<answer>\s*(-?\d+)\s*</answer>', output)
    if len(m) >= 1 and int(m[0]) == gold:
        rew += 1.0
    n_think = len(re.findall(r'<think>', output)) == 1 and len(re.findall(r'</think>', output)) == 1
    n_ans = len(re.findall(r'<answer>', output)) == 1 and len(re.findall(r'</answer>', output)) == 1
    if n_think and n_ans:
        rew += format_bonus
    return rew

In [ ]:
# 练习 2 参考答案
def group_advantage(rewards, normalize=False, eps=1e-8):
    rewards = np.asarray(rewards, dtype=float)
    adv = rewards - rewards.mean()
    if normalize:
        adv = adv / (rewards.std() + eps)
    return adv

In [ ]:
# 练习 3 参考答案
def temperature_sample(logits_1d, temp, r):
    p = softmax(np.asarray(logits_1d, dtype=float)[None], temp)[0]
    return int(r.choice(len(p), p=p))

In [ ]:
# 练习 4 参考答案
def reinforce_step(logits, trajs, rewards, lr=0.3, use_baseline=True):
    rewards = np.asarray(rewards, dtype=float)
    b = rewards.mean() if use_baseline else 0.0
    grad = np.zeros_like(logits)
    for traj, R in zip(trajs, rewards):
        grad += (R - b) * dlogp_dlogits(logits, traj)
    grad /= len(trajs)
    return logits + lr * grad

---
## 🧪 真实数据胶囊：GSM8K 风格的可验证奖励

用 **GSM8K** 的真实约定（最终答案写在 `####` 之后，含 `<<算式=结果>>` 步骤标注）构造一个真实的结果奖励管线，并在几条**真实 GSM8K 格式**的样例上判分。带 try/except：联网失败则回退到内置的真实 GSM8K 样例。

In [ ]:
import re
# 尝试联网取真实 GSM8K；失败则用内置真实样例(取自 GSM8K train，格式原样)
GSM8K_FALLBACK = [
    {'question': 'Natalia sold clips to 48 friends in April, and half as many in May. How many altogether?',
     'answer': 'In May she sold 48/2 = <<48/2=24>>24 clips.\nAltogether 48+24 = <<48+24=72>>72 clips.\n#### 72'},
    {'question': 'Weng earns $12 an hour. Yesterday she did 50 minutes. How much did she earn?',
     'answer': 'Per minute 12/60 = $<<12/60=0.2>>0.2.\n50 minutes => 0.2 x 50 = $<<0.2*50=10>>10.\n#### 10'},
]
try:
    from datasets import load_dataset
    ds = load_dataset('openai/gsm8k', 'main', split='train[:2]')
    samples = [{'question': r['question'], 'answer': r['answer']} for r in ds]
    print('已联网载入真实 GSM8K 样例')
except Exception as e:
    samples = GSM8K_FALLBACK
    print('离线回退到内置真实 GSM8K 样例 (', type(e).__name__, ')')

def gsm8k_gold(answer_field):
    '''GSM8K 标准答案：#### 后的数字。'''
    return int(re.findall(r'####\s*(-?[0-9][0-9,]*)', answer_field)[-1].replace(',', ''))

for s in samples:
    gold = gsm8k_gold(s['answer'])
    print(f'题: {s["question"][:50]}...  标准答案 = {gold}')
    assert isinstance(gold, int)
print('✅ 真实 GSM8K 可验证奖励：从 #### 抽出 ground-truth，用于 ORM/RL 判分')

**🧪 胶囊练习**：实现 `gsm8k_outcome_reward(model_output, answer_field)` —— 从模型输出抽 `#### 数字`、与 GSM8K 标准答案比对，返回 1.0/0.0。这就是 R1 在数学上用的规则奖励。

In [ ]:
def gsm8k_outcome_reward(model_output, answer_field):
    # TODO: gold = gsm8k_gold(answer_field); 从 model_output 抽 #### 数字; 相等 ->1 else 0
    raise NotImplementedError

In [ ]:
# 自测
ans = samples[0]['answer']
gold = gsm8k_gold(ans)
assert gsm8k_outcome_reward(f'... so the total is #### {gold}', ans) == 1.0
assert gsm8k_outcome_reward('#### 999999', ans) == 0.0
assert gsm8k_outcome_reward('no final answer here', ans) == 0.0
print('✅ 胶囊练习通过：真实 GSM8K 规则奖励管线')

In [ ]:
# 📖 胶囊参考答案
def gsm8k_outcome_reward(model_output, answer_field):
    gold = gsm8k_gold(answer_field)
    m = re.findall(r'####\s*(-?[0-9][0-9,]*)', model_output)
    if not m:
        return 0.0
    return 1.0 if int(m[-1].replace(',', '')) == gold else 0.0

### 小结
- **结果奖励 RL**：只在末端给 0/1 可验证奖励，用策略梯度推高『导向正确答案』的生成方式。
- **策略梯度** $\nabla J=\mathbb{E}[R\nabla\log\pi]$：高回报轨迹整条被强化；我们对拍数值梯度验证正确。
- **基线**(减组均值)：无偏地大幅降方差 —— 稀疏奖励能否训稳的分水岭，也是 GRPO 优势的雏形。
- **温度**：探索-利用旋钮；RL 须防**熵塌缩**(过早收敛、停止探索)，熵奖励可缓解。
- **format 奖励**：温和的奖励塑形(让答案可抽取)；但直接奖励长度/反思词会被 hack —— 别优化可被 hack 的代理。
- **长 CoT 涌现**：仅凭对错奖励，『会检查/多想』因提高正确率被自然选择出来；检查无效则不涌现。

下一站：**模块 02 · GRPO 从零** —— 把『组均值基线』正式升级为 R1 真正使用的 **组相对策略优化**。